In [ ]:
# Supprimer le dossier de cache pour NeuroDSL
cache_dir = joinpath(homedir(), ".julia", "compiled", "v1.10", "NeuroDSL")
if isdir(cache_dir)
    rm(cache_dir; recursive=true, force=true)
    println("Cache supprimé : $cache_dir")
else
    println("Pas de cache trouvé")
end

In [ ]:
include(raw"C:\Users\Nevermind\Desktop\NeuroDSL\src\NeuroDSL.jl")
using .NeuroDSL

In [2]:
using NeuroDSL

# 1. ENREGISTRER LES OPÉRATEURS (obligatoire avant toute utilisation)
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2])
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

# 2. Construction du graphe
g = NeuroGraph(namespace=:fib, device=Backend.CPUDevice())

builder = @neuro g ns=:fib begin
    @node x0 = 1.0
    @node x1 = 2.0
    @rule fib(n::Int) = n <= 1 ? Symbol(:x, n) : wsum(weights=[1.0, 1.0], fib(n-1), fib(n-2))
    @node f8 = fib(8)
    even_terms = [fib(i) for i in 0:2:8]
    @node sn = nsum(even_terms...)
end

# 3. Exécution forward
log = ExecutionLog()
demand!(g, :sn; ctx_store=CtxStore(), namespace=:fib, log=log)

# 4. Visualisation interactive
save_interactive_graph(g, log, "fib_stream.html"; title="Fibonacci Stream")

✅ Op :wsum enregistré
✅ Op :nsum enregistré
✅ Op :identity enregistré
✅ Interactive Trace exporté → fib_stream.html


In [ ]:
# V2
using NeuroDSL

# ── Définition de tous les opérateurs avec @defop ────────────────────
@defop identity out = inputs[1]
@defop wsum     out = w1 * inputs[1] + w2 * inputs[2]

@defop nsum begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

# ── Construction du graphe ───────────────────────────────────────────
g = NeuroGraph(namespace=:fib, device=Backend.CPUDevice())

builder = @neuro g ns=:fib begin
    @node x0 = 1.0
    @node x1 = 2.0
    @rule fib(n::Int) = n <= 1 ? Symbol(:x, n) : wsum(w1=1.0, w2=1.0, fib(n-1), fib(n-2))
    @node f8 = fib(8)
    even_terms = [fib(i) for i in 0:2:8]
    @node sn = nsum(even_terms...)
end

# ── Exécution ────────────────────────────────────────────────────────
log = ExecutionLog()
demand!(g, :sn; ctx_store=CtxStore(), namespace=:fib, log=log)

# ── Visualisation ────────────────────────────────────────────────────
save_interactive_graph(g, log, "fib_stream.html"; title="Fibonacci Stream")

In [4]:
using NeuroDSL

# ── 1. Enregistrement des opérateurs ─────────────────────────────────────
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2])
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

# ── 2. Construction déclarative du graphe ────────────────────────────────
g = NeuroGraph(namespace=:vec, device=Backend.CPUDevice())

builder = @neuro g ns=:vec begin
    # Vecteurs initiaux (taille 2)
    @node v0 = [1.0, 0.0]
    @node v1 = [0.0, 1.0]

    # Règle récursive vectorielle : v_n = 0.5*v_{n-1} + 0.5*v_{n-2}
    @rule vec_seq(n::Int) = n <= 1 ? Symbol(:v, n) : wsum(weights=[0.5, 0.5], vec_seq(n-1), vec_seq(n-2))

    # Calculer les 5 premiers termes (v2 à v5)
    @node v2 = vec_seq(2)
    @node v3 = vec_seq(3)
    @node v4 = vec_seq(4)
    @node v5 = vec_seq(5)

    # Moyenne des 3 derniers vecteurs (v3, v4, v5)
    @node avg = nsum(v3, v4, v5)   # sera divisé par 3 dans un second temps ? ou utiliser un opérateur scale
end

# ── 3. Exécution forward ─────────────────────────────────────────────────
log = ExecutionLog()
demand!(g, :avg; ctx_store=CtxStore(), namespace=:vec, log=log)

# Afficher les valeurs finales
println("v0 = ", Array(g.nodes[:vec][:v0].value))
println("v1 = ", Array(g.nodes[:vec][:v1].value))
println("v5 = ", Array(g.nodes[:vec][:v5].value))
println("avg = ", Array(g.nodes[:vec][:avg].value))

# ── 4. Visualisation interactive ─────────────────────────────────────────
save_interactive_graph(g, log, "vec_stream.html"; title="Vector Stream")

✅ Op :wsum enregistré
✅ Op :nsum enregistré
✅ Op :identity enregistré
v0 = Float32[1.0, 0.0]
v1 = Float32[0.0, 1.0]
v5 = Float32[0.3125, 0.6875]
avg = Float32[0.9375, 2.0625]
✅ Interactive Trace exporté → vec_stream.html


In [5]:
using NeuroDSL

# Opérateur générique : wsum avec un vecteur de poids
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        w = attrs[:weights]
        @. out = w[1] * inputs[1]
        for i in 2:length(w)
            @. out += w[i] * inputs[i]
        end
    end
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

g = NeuroGraph(namespace=:vec, device=Backend.CPUDevice())

builder = @neuro g ns=:vec begin
    # Feuilles vectorielles nommées v0, v1, v2
    @node v0 = [1.0, 0.0, 0.0]
    @node v1 = [0.0, 1.0, 0.0]
    @node v2 = [0.0, 0.0, 1.0]

    # Règle Tribonacci vectorielle
    @rule tribo(n::Int) = n <= 2 ? Symbol(:v, n) : wsum(weights=[0.4, 0.3, 0.3], tribo(n-1), tribo(n-2), tribo(n-3))

    # Générer les termes suivants
    @node t3 = tribo(3)
    @node t4 = tribo(4)
    @node t5 = tribo(5)

    # Somme des trois
    @node total = nsum(t3, t4, t5)
end

log = ExecutionLog()
NeuroDSL.demand!(g, :total; ctx_store=CtxStore(), namespace=:vec, log=log)

println("t3 = ", Array(NeuroDSL.node(g, :t3; namespace=:vec).value))
println("t4 = ", Array(NeuroDSL.node(g, :t4; namespace=:vec).value))
println("t5 = ", Array(NeuroDSL.node(g, :t5; namespace=:vec).value))
println("total = ", Array(NeuroDSL.node(g, :total; namespace=:vec).value))

save_interactive_graph(g, log, "tribo_stream.html"; title="Tribonacci Vector Stream")

✅ Op :wsum enregistré
✅ Op :nsum enregistré
✅ Op :identity enregistré
t3 = Float32[0.3, 0.3, 0.4]
t4 = Float32[0.120000005, 0.42000002, 0.46]
t5 = Float32[0.13800001, 0.25800002, 0.604]
total = Float32[0.558, 0.97800004, 1.464]
✅ Interactive Trace exporté → tribo_stream.html


In [6]:
# 1. Enregistrer les opérateurs personnalisés
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = 0.3f0 * inputs[1] + 0.7f0 * inputs[2])
)

NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        @inbounds for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)

# 2. Construction du graphe Fibonacci
function build_fib_graph!(g, n; ns=:fib, vec_size=10)
    NeuroDSL.set!(g, :f0, randn(Float32, vec_size); namespace=ns)
    NeuroDSL.set!(g, :f1, randn(Float32, vec_size); namespace=ns)

    even_nodes = Symbol[:f0]        # f[0] est pair (indice 0)
    prev0, prev1 = :f0, :f1

    for i in 0:n-1
        f2 = Symbol(:f, i + 2)
        NeuroDSL.addrule!(g,
            NeuroDSL.GraphRule(f2, [prev0, prev1], :wsum; namespace=ns))
        prev0, prev1 = prev1, f2
        # i impair → f2 est à un indice pair
        i % 2 == 1 && push!(even_nodes, f2)
    end

    yn_sym = prev1      # dernier terme = f[n+1]

    # Nœud large : somme des termes pairs
    NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(:sn, even_nodes, :nsum; namespace=ns))

    return yn_sym, :sn
end

# 3. Exécution et benchmarking
n = 10
println("="^58)
@printf " NeuroDSL  Fibonacci DAG  n = %d\n" n
println("="^58)

t_build = @elapsed begin
    g = NeuroDSL.NeuroGraph(namespace=:fib, device=NeuroDSL.Backend.CPUDevice())
    yn_sym, sn_sym = build_fib_graph!(g, n; ns=:fib)
end
@printf "  Construction   %6d nœuds     %.4f s\n" (n+3) t_build

# Log d'exécution pour la visualisation
log = ExecutionLog()
ctx = CtxStore()

# Calcul de la somme des termes pairs (sn) et du dernier terme (yn)
NeuroDSL.invalidate_all!(g; namespace=:fib)
NeuroDSL.demand!(g, sn_sym; ctx_store=ctx, namespace=:fib, log=log)
NeuroDSL.demand!(g, yn_sym; ctx_store=ctx, namespace=:fib, log=log)

# Affichage des résultats
sn_val = Array(NeuroDSL.node(g, :sn; namespace=:fib).value)
yn_val = Array(NeuroDSL.node(g, yn_sym; namespace=:fib).value)
@printf "  sn (wide)  : [%.4f, %.4f, ...]\n" sn_val[1] sn_val[2]
@printf "  yn (deep)  : [%.4f, %.4f, ...]\n" yn_val[1] yn_val[2]

# Génération du fichier HTML interactif
save_interactive_graph(g, log, "fibonacci_n10.html"; title="Fibonacci DAG (n=10)")

println("\n✅ Visualisation sauvegardée dans fibonacci_n10.html")

✅ Op :wsum enregistré


✅ Op :nsum enregistré
 NeuroDSL  Fibonacci DAG  n = 10
  Construction       13 nœuds     0.0508 s
  sn (wide)  : [6.4185, -0.1049, ...]
  yn (deep)  : [1.2144, -0.0024, ...]
✅ Interactive Trace exporté → fibonacci_n10.html

✅ Visualisation sauvegardée dans fibonacci_n10.html


In [1]:
# Supprimer le cache de précompilation
cache_dir = joinpath(homedir(), ".julia", "compiled", "v1.10", "NeuroDSL")
isdir(cache_dir) && rm(cache_dir; recursive=true, force=true)

# Charger le module avec include pour voir l'erreur entière
include(raw"C:\Users\Nevermind\Desktop\NeuroDSL\src\NeuroDSL.jl")

Main.NeuroDSL

In [1]:
using NeuroDSL, Printf

# ── 1. Enregistrement des opérateurs (version stable) ─────────────────────
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2])
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

# ── 2. Construction du graphe déclaratif + liste dynamique externe ────────
function build_fib_graph(n::Int)
    g = NeuroGraph(namespace=:fib, device=Backend.CPUDevice())
    
    # Bloc @neuro (sans aucune compréhension de liste)
    builder = @neuro g ns=:fib begin
        @node f0 = randn(Float32, 10)
        @node f1 = randn(Float32, 10)
        @rule fib(n::Int) = n <= 1 ? Symbol(:f, n) : wsum(weights=[0.3, 0.7], fib(n-1), fib(n-2))
        @node yn = fib(n+1)
    end

    # Construction dynamique des termes pairs EN DEHORS de la macro
    even_terms = Symbol[call_rule(builder, :fib, i) for i in 0:1:n]   # Vector{Symbol} garanti
    addrule!(builder.graph, GraphRule(:sn, even_terms, :nsum; namespace=builder.namespace))

    return builder
end

# ── 3. Exécution forward ───────────────────────────────────────────────────
n = 10
builder = build_fib_graph(n)
g = builder.graph

log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :sn; ctx_store=ctx, namespace=:fib, log=log)
NeuroDSL.demand!(g, :yn; ctx_store=ctx, namespace=:fib, log=log)

sn_val = Array(NeuroDSL.node(g, :sn; namespace=:fib).value)
yn_val = Array(NeuroDSL.node(g, :yn; namespace=:fib).value)
@printf "  sn (wide) : [%.4f, %.4f, ...]\n" sn_val[1] sn_val[2]
@printf "  yn (deep) : [%.4f, %.4f, ...]\n" yn_val[1] yn_val[2]

# ── 4. Visualisation interactive ───────────────────────────────────────────
save_interactive_graph(g, log, "fibonacci_rec_n10.html"; title="Fibonacci Récursif (n=10)")
println("\n✅ Visualisation sauvegardée dans fibonacci_rec_n10.html")

┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   cuSPARSE [b26da814-b3bc-49ef-b0ee-c816305aa060]
│   ChainRulesCoreExt [8ccf546c-b5ad-5491-b9b6-b62e2a4db961]
│   GPUArraysSparseArraysExt [555d850b-e4ac-5d9d-bfa0-17bb432b4efb]
│   ChainRules [082447d4-558c-5d27-93f4-14fc19e9eca2]
│   NVML [611af6d1-644e-4c5d-bd58-854d7d1254b9]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   JLD2Ext [3dbd0623-ce14-52d8-8601-bc177a3b211d]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   ZygoteExt [3f48baa7-5843-56cc-8004-b8b725de387b]
│   FluxCUDAExt [dd41ee52-2073-581e-92e8-26baf003f19a]
│   cuSOLVER [887afef0-6a32-4de5-add4-7827692ba8fc]
│   CUDAExt [9e0aca61-9665-56cf-9642-d947ef6fc392]
│   StructArraysAdaptExt [f04e5bcb-ab32-5a64-8b64-c2cc4abec66e]
│   OneHotArrays [0b1bfda6-eb8a-41d2-88d8-f5af5cad476f]
│   CUDACore [bd0ed864-bdfe-4181-a5ed-ce625a5fdea2]
│   ZygoteDistancesExt [5865c103-18d1-586a-9b11-010bbc2260a8]
│   MLUtilsExt [447e3217-c189

✅ Op :wsum enregistré
✅ Op :nsum enregistré
✅ Op :identity enregistré
  sn (wide) : [-6.7697, 9.9027, ...]
  yn (deep) : [-0.6796, 0.9369, ...]
✅ Interactive Trace exporté → fibonacci_rec_n10.html

✅ Visualisation sauvegardée dans fibonacci_rec_n10.html


In [4]:
using NeuroDSL, Printf

# ── 1. Opérateurs nécessaires ───────────────────────────────────────────
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2])
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

# ── 2. Graphe déclaratif ────────────────────────────────────────────────
n = 6
g = NeuroGraph(namespace=:comb, device=Backend.CPUDevice())

builder = @neuro g ns=:comb begin
    # Cas de base C(n,0)=1, C(n,n)=1
    @node one = 1.0

    # Règle récursive de Pascal
    @rule comb(n::Int, k::Int) = (k == 0 || k == n) ? :one :
                                 wsum(weights=[1.0, 1.0], comb(n-1, k-1), comb(n-1, k))

    # Somme sur k pair de 0 à n
    even_ks = [comb(n, k) for k in 1:2:n]
    @node total = nsum(even_ks...)
end

# ── 3. Exécution forward ────────────────────────────────────────────────
log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :total; ctx_store=ctx, namespace=:comb, log=log)

# ── 4. Affichage du résultat ────────────────────────────────────────────
val = Array(NeuroDSL.node(g, :total; namespace=:comb).value)
println("Somme des C(10, k) pour k pair = $(val[])")   # 512.0 (c'est 2^(n-1) = 512)

# ── 5. Visualisation interactive ────────────────────────────────────────
save_interactive_graph(g, log, "comb_even_sum.html"; title="Somme C(10,k) pour k pair")

✅ Op :wsum enregistré
✅ Op :nsum enregistré
✅ Op :identity enregistré
Somme des C(10, k) pour k pair = 32.0
✅ Interactive Trace exporté → comb_even_sum.html


In [2]:
using NeuroDSL, Printf

NeuroDSL.register_op!(:scale_add,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        factor = attrs[:factor]
        @. out = factor * inputs[1] + inputs[2]
    end
)

register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)

register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)
g = NeuroGraph(namespace=:stirling, device=Backend.CPUDevice())

builder = @neuro g ns=:stirling operators=[:scale_add] begin
    @node one  = 1.0
    @node zero = 0.0

    @rule stirling(n::Int, k::Int) = begin
        if n == 0 && k == 0
            :one
        elseif k == 0 || k > n
            :zero
        else
            scale_add(factor=k, stirling(n-1, k), stirling(n-1, k-1))
        end
    end

    @node s53 = stirling(5, 3)
    all_terms = [stirling(5, k) for k in 1:5]
    @node bell5 = nsum(all_terms...)
end


# ── 3. Exécution ───────────────────────────────────────────────────────────
log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :s53; ctx_store=ctx, namespace=:stirling, log=log)
NeuroDSL.demand!(g, :bell5; ctx_store=ctx, namespace=:stirling, log=log)

s53_val = Array(NeuroDSL.node(g, :s53; namespace=:stirling).value)[]
bell5_val = Array(NeuroDSL.node(g, :bell5; namespace=:stirling).value)[]
println("S(5,3) = $(s53_val)")   # doit être 25
println("Somme S(5,k) pour k=1..5 = $(bell5_val) (B₅ = 52)")

# ── 4. Visualisation ───────────────────────────────────────────────────────
save_interactive_graph(g, log, "stirling_example.html"; title="Nombres de Stirling S(5,3)")
println("✅ Visualisation sauvegardée dans stirling_example.html")

✅ Op :scale_add enregistré
✅ Op :nsum enregistré
✅ Op :identity enregistré
S(5,3) = 25.0
Somme S(5,k) pour k=1..5 = 52.0 (B₅ = 52)
✅ Interactive Trace exporté → stirling_example.html
✅ Visualisation sauvegardée dans stirling_example.html
